# Transcode BigWig Signal Analysis

Extracts per-nucleotide Ribo-seq coverage from strand-specific bigwig files for ORFs
in the translon DB, and scores them for mean coverage, 3-nt periodicity, and start-peak ratio.

Requires `pyBigWig`. Install with: `pip install pyBigWig` or `conda install -c bioconda pybigwig`

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── pyBigWig import (optional — graceful failure) ──────────────────────────
try:
    import pyBigWig
    HAS_PYBIGWIG = True
    print("pyBigWig available:", pyBigWig.__version__ if hasattr(pyBigWig, '__version__') else 'yes')
except ImportError:
    HAS_PYBIGWIG = False
    print("WARNING: pyBigWig not installed.")
    print("  Install with: pip install pyBigWig")
    print("  Signal extraction cells will be skipped.")

# ── Paths ──────────────────────────────────────────────────────────────────
# HPC path (default)
DB = Path("/hps/nobackup/flicek/ensembl/genebuild/jackt/riboseq/pilot/full_pilot_results/translon_db_rebuild/translon_db/translons.sqlite")
# Local fallback: DB = Path("/path/to/local/translons.sqlite")

BIGWIG_ROOT = Path("/hps/nobackup/flicek/ensembl/genebuild/jackt/riboseq/pilot/bigwigs")
# riboseqorg-nf produces: {sample_id}.forward.bw and {sample_id}.reverse.bw

OUT_DIR = Path("figures_bigwig")
OUT_DIR.mkdir(exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────────────
TOOL_ORDER = ["PRICE", "RiboTIE", "ORFQuant", "iRibo", "RibORF2"]
CLS_ORDER  = ["cds", "non_cds"]
CLS_LABELS = {"cds": "CDS", "non_cds": "Non-CDS"}
TOOL_COLORS = {
    "PRICE":    "#4C72B0",
    "RiboTIE":  "#DD8452",
    "ORFQuant": "#55A868",
    "iRibo":    "#C44E52",
    "RibORF2":  "#8172B2",
}
CLS_COLORS = {"cds": "#2196F3", "non_cds": "#FF9800"}

# ── Expected samples ───────────────────────────────────────────────────────
EXPECTED_SAMPLES = [
    # Pancreas
    "SRR11005875_to_79", "SRR11005880_to_84", "SRR11005885_to_89",
    "SRR11005890_to_94", "SRR11005895_to_99", "SRR11005900_to_04",
    "Ribo_Pancreas_pooled",
    # Fibroblast
    "SRR15513179", "SRR15513180", "SRR15513181", "SRR15513182",
    "Fib_24_45m", "Fib_24_bsl", "Fib_27_45m", "Fib_27_bsl",
    "Fib_41_45m", "Fib_41_bsl", "Ribo_Fib_pooled",
    # Endothelial
    "SRR15513197", "SRR15513198_GENELAB1026", "SRR15513199",
    "SRR15513200", "SRR15513201", "SRR15513202_GENELAB1143",
    "Ribo_EC_pooled",
]

print(f"DB path:         {DB}")
print(f"DB exists:       {DB.exists()}")
print(f"Bigwig root:     {BIGWIG_ROOT}")
print(f"Bigwig root exists: {BIGWIG_ROOT.exists()}")
print(f"Expected samples: {len(EXPECTED_SAMPLES)}")


# ── BigWig discovery ───────────────────────────────────────────────────────
def discover_bigwigs(root: Path) -> pd.DataFrame:
    """
    Discover strand-specific bigwig files under `root`.

    Globs for common suffix patterns produced by riboseqorg-nf:
        {sample_id}.forward.bw   {sample_id}.reverse.bw
        {sample_id}.fwd.bw       {sample_id}.rev.bw
        {sample_id}_forward.bw   {sample_id}_reverse.bw

    Returns a DataFrame with columns:
        sample_id, fwd_path, rev_path, has_pair (bool)
    """
    if not root.exists():
        print(f"WARNING: bigwig root does not exist yet: {root}")
        print("         Returning empty manifest — run signal cells after bigwigs are available.")
        return pd.DataFrame(columns=['sample_id', 'fwd_path', 'rev_path', 'has_pair'])

    FWD_SUFFIXES = [".forward.bw", ".fwd.bw", "_forward.bw"]
    REV_SUFFIXES = [".reverse.bw", ".rev.bw", "_reverse.bw"]

    def strip_suffix(stem: str, suffixes: list) -> str | None:
        for suf in suffixes:
            # suf includes the dot/underscore — match against stem + suf = full filename w/o final .bw
            pass
        return None  # handled below via filename

    fwd_map: dict[str, Path] = {}
    rev_map: dict[str, Path] = {}

    for bw in root.rglob("*.bw"):
        name = bw.name
        matched = False
        for suf in FWD_SUFFIXES:
            if name.endswith(suf):
                sample_id = name[: -len(suf)]
                fwd_map[sample_id] = bw
                matched = True
                break
        if not matched:
            for suf in REV_SUFFIXES:
                if name.endswith(suf):
                    sample_id = name[: -len(suf)]
                    rev_map[sample_id] = bw
                    break

    all_samples = set(fwd_map) | set(rev_map)
    rows = []
    for sid in sorted(all_samples):
        fwd = fwd_map.get(sid)
        rev = rev_map.get(sid)
        rows.append({
            'sample_id': sid,
            'fwd_path':  str(fwd) if fwd else None,
            'rev_path':  str(rev) if rev else None,
            'has_pair':  fwd is not None and rev is not None,
        })

    df = pd.DataFrame(rows)
    print(f"\nBigWig discovery summary:")
    print(f"  Total samples found:        {len(df)}")
    print(f"  With both strands (pair):   {df['has_pair'].sum()}")
    print(f"  Missing one strand:         {(~df['has_pair']).sum()}")

    # Flag which expected samples are present / missing
    found_set   = set(df['sample_id'])
    missing     = [s for s in EXPECTED_SAMPLES if s not in found_set]
    unexpected  = [s for s in found_set if s not in EXPECTED_SAMPLES]
    print(f"\n  Expected samples with bigwigs: {len(EXPECTED_SAMPLES) - len(missing)} / {len(EXPECTED_SAMPLES)}")
    if missing:
        print(f"  Missing: {missing}")
    if unexpected:
        print(f"  Unexpected (not in expected list): {unexpected}")

    return df


bw_manifest = discover_bigwigs(BIGWIG_ROOT)
print()
print(bw_manifest)

## Signal extraction utilities

Functions to extract per-nucleotide coverage from strand-specific bigwigs over spliced ORF intervals.

In [ ]:
def _parse_bed12_blocks(bed_start: int, block_sizes_str: str,
                        block_starts_str: str) -> list[tuple[int, int]]:
    """
    Parse BED12 block_sizes / block_starts strings into a list of
    (chrom_start, chrom_end) intervals (0-based, half-open).
    """
    sizes  = [int(x) for x in block_sizes_str.rstrip(',').split(',') if x]
    starts = [int(x) for x in block_starts_str.rstrip(',').split(',') if x]
    intervals = []
    for s, sz in zip(starts, sizes):
        ivl_start = bed_start + s
        intervals.append((ivl_start, ivl_start + sz))
    return intervals


def extract_coverage(
    bw_fwd: str,
    bw_rev: str,
    chrom: str,
    strand: str,
    intervals: list[tuple[int, int]],
) -> np.ndarray | None:
    """
    Extract per-nucleotide Ribo-seq coverage over spliced intervals from
    strand-specific bigwigs.

    Parameters
    ----------
    bw_fwd, bw_rev : str
        Paths to forward- and reverse-strand bigwig files.
    chrom : str
        Chromosome name (must match bigwig chromosome names).
    strand : str
        '+' or '-'.
    intervals : list of (start, end)
        Genomic intervals in 0-based half-open coordinates (BED-style),
        sorted in ascending genomic order.

    Returns
    -------
    np.ndarray
        1D array of per-nucleotide coverage in 5'→3' RNA order
        (length = sum of interval lengths). NaN replaced with 0.
        Returns None if pyBigWig is unavailable, the chromosome is missing,
        or an IO error occurs.
    """
    if not HAS_PYBIGWIG:
        return None

    bw_path = bw_fwd if strand == '+' else bw_rev
    if bw_path is None:
        return None

    try:
        bw = pyBigWig.open(str(bw_path))
        chrom_sizes = bw.chroms()
        if chrom not in chrom_sizes:
            bw.close()
            return None

        chunks = []
        chrom_len = chrom_sizes[chrom]
        for start, end in intervals:
            # Clamp to chromosome bounds
            start = max(0, start)
            end   = min(end, chrom_len)
            if start >= end:
                continue
            vals = bw.values(chrom, start, end, numpy=True)
            if vals is None:
                vals = np.zeros(end - start, dtype=np.float32)
            else:
                vals = np.where(np.isnan(vals), 0.0, vals)
            chunks.append(vals.astype(np.float32))

        bw.close()

        if not chunks:
            return None

        arr = np.concatenate(chunks)
        # Reverse to 5'→3' RNA order for minus-strand features
        if strand == '-':
            arr = arr[::-1]
        return arr

    except Exception as exc:
        # Silently return None on IO errors (missing file, corrupt bw, etc.)
        return None


print("extract_coverage() defined.")
print("Signature: extract_coverage(bw_fwd, bw_rev, chrom, strand, intervals) -> np.ndarray | None")

## Scoring functions

Three metrics derived from the per-nucleotide coverage array:

- **Mean coverage**: average reads per nucleotide.
- **Periodicity score**: fraction of signal in frame-0 (3-nt periodicity).
- **Start-peak ratio**: enrichment of coverage at the 5' end relative to the body.

In [ ]:
def mean_coverage(arr: np.ndarray) -> float:
    """Mean per-nucleotide coverage."""
    if arr is None or len(arr) == 0:
        return np.nan
    return float(np.mean(arr))


def periodicity_score(arr: np.ndarray) -> float:
    """
    3-nt periodicity score: fraction of total signal falling in frame 0
    (positions 0, 3, 6, ...), computed by summing each reading frame.

    Returns np.nan for arrays shorter than 9 nt or with zero total.
    Range: [0, 1]. Random expectation ≈ 0.33.
    """
    if arr is None or len(arr) < 9:
        return np.nan

    # Pad to multiple of 3
    pad = (3 - len(arr) % 3) % 3
    if pad:
        arr = np.concatenate([arr, np.zeros(pad, dtype=arr.dtype)])

    arr_2d = arr.reshape(-1, 3)   # shape (n_codons, 3)
    frame_sums = arr_2d.sum(axis=0)  # sum per frame position
    total = frame_sums.sum()

    if total == 0:
        return np.nan
    return float(frame_sums[0] / total)


def start_peak_ratio(arr: np.ndarray, window: int = 15) -> float:
    """
    Ratio of mean coverage in the first `window` nucleotides (5' start region)
    relative to mean coverage in the remainder of the ORF.

    Values > 1 indicate start-peak enrichment (typical of Ribo-seq start-codon pileup).
    Returns np.nan if the array is too short or the body is empty / zero.
    """
    if arr is None or len(arr) < window + 3:
        return np.nan

    start_mean = np.mean(arr[:window])
    body_mean  = np.mean(arr[window:])

    if body_mean == 0:
        return np.nan if start_mean == 0 else np.inf
    return float(start_mean / body_mean)


# Quick unit tests
test_arr = np.array([10, 0, 0, 8, 0, 0, 6, 0, 0, 4, 0, 0], dtype=float)
assert abs(periodicity_score(test_arr) - 1.0) < 1e-6, "Perfect frame-0 should score 1.0"
flat_arr = np.ones(12, dtype=float)
assert abs(periodicity_score(flat_arr) - 1/3) < 1e-6, "Flat signal should score ~0.333"
print("Scoring functions defined and self-tests passed.")
print(f"  periodicity_score(perfect frame-0): {periodicity_score(test_arr):.3f}")
print(f"  periodicity_score(flat):            {periodicity_score(flat_arr):.3f}")

## Score a sample of ORFs

Load ORFs from the DB, match each to its sample's bigwig pair, extract coverage,
and compute the three signal metrics.

Set `LIMIT = None` to score all ORFs (production mode).

In [ ]:
LIMIT = 5000  # Set to None for full production run

limit_clause = f"LIMIT {LIMIT}" if LIMIT is not None else ""

con = sqlite3.connect(DB)
orfs = pd.read_sql_query(f"""
    SELECT DISTINCT
           source_tool,
           source_feature_class AS cls,
           feature_key,
           bed_chrom,
           bed_start,
           bed_end,
           bed_strand,
           block_sizes,
           block_starts,
           spliced_length_nt,
           sample_id
    FROM translons
    WHERE qc_status = 'pass'
      AND source_feature_class IN ('cds', 'non_cds')
      AND sample_id NOT GLOB '*_fastq'
    {limit_clause}
""", con)
con.close()

print(f"ORFs loaded: {len(orfs):,}")
print(orfs.head())

# Build a per-sample bigwig lookup dict: sample_id → (fwd_path, rev_path)
bw_lookup: dict[str, tuple[str | None, str | None]] = {}
if not bw_manifest.empty:
    for _, row in bw_manifest.iterrows():
        bw_lookup[row['sample_id']] = (row['fwd_path'], row['rev_path'])

print(f"\nBigwig pairs available for {len(bw_lookup)} samples.")

if not HAS_PYBIGWIG:
    print("\nSkipping signal extraction: pyBigWig not installed.")
    scored = pd.DataFrame()
elif bw_manifest.empty:
    print("\nSkipping signal extraction: no bigwig files found.")
    scored = pd.DataFrame()
else:
    records = []
    n_no_bw   = 0
    n_no_cov  = 0
    n_scored  = 0

    for i, row in orfs.iterrows():
        sid = row['sample_id']
        if sid not in bw_lookup:
            n_no_bw += 1
            continue

        fwd_path, rev_path = bw_lookup[sid]

        try:
            intervals = _parse_bed12_blocks(
                int(row['bed_start']),
                str(row['block_sizes']),
                str(row['block_starts']),
            )
        except Exception:
            n_no_cov += 1
            continue

        arr = extract_coverage(
            fwd_path, rev_path,
            str(row['bed_chrom']),
            str(row['bed_strand']),
            intervals,
        )

        if arr is None:
            n_no_cov += 1
            continue

        records.append({
            'feature_key':    row['feature_key'],
            'source_tool':    row['source_tool'],
            'cls':            row['cls'],
            'sample_id':      sid,
            'mean_cov':       mean_coverage(arr),
            'periodicity':    periodicity_score(arr),
            'start_peak_ratio': start_peak_ratio(arr),
        })
        n_scored += 1

        if n_scored % 500 == 0:
            print(f"  Scored {n_scored:,} ORFs...", end='\r')

    scored = pd.DataFrame(records)
    print(f"\nScoring complete:")
    print(f"  Scored:              {n_scored:,}")
    print(f"  No bigwig for sample:{n_no_bw:,}")
    print(f"  No coverage/error:   {n_no_cov:,}")
    print(scored.describe())

## Signal plots

Distributions of mean coverage, periodicity, and start-peak ratio by tool and ORF class.

In [ ]:
if scored.empty:
    print("No scored ORFs available — skipping coverage distribution plot.")
else:
    tools_present = [t for t in TOOL_ORDER if t in scored['source_tool'].values]

    def violin_log(ax, data_list, positions, colors, title, ylabel):
        """Draw violin plots of log10-transformed data."""
        for pos, data, color in zip(positions, data_list, colors):
            data_clean = np.asarray(data, dtype=float)
            data_clean = data_clean[np.isfinite(data_clean) & (data_clean > 0)]
            if len(data_clean) < 5:
                ax.scatter([pos], [np.nan], color=color)
                continue
            log_data = np.log10(data_clean)
            parts = ax.violinplot([log_data], positions=[pos], widths=0.7,
                                  showmedians=True, showextrema=False)
            for pc in parts['bodies']:
                pc.set_facecolor(color)
                pc.set_alpha(0.7)
            parts['cmedians'].set_color('black')
            parts['cmedians'].set_linewidth(2)
        ax.set_xticks(list(range(1, len(positions) + 1)))
        ax.set_xticklabels(tools_present, rotation=30, ha='right')
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.yaxis.grid(True, alpha=0.4)
        ax.set_axisbelow(True)

    fig, axes = plt.subplots(1, len(CLS_ORDER), figsize=(13, 6))
    for ax, cls in zip(axes, CLS_ORDER):
        sub = scored[(scored['cls'] == cls) & (scored['mean_cov'] > 0)]
        data_list = [sub[sub['source_tool'] == t]['mean_cov'].dropna().values
                     for t in tools_present]
        colors = [TOOL_COLORS.get(t, '#888') for t in tools_present]
        violin_log(ax, data_list, list(range(1, len(tools_present) + 1)), colors,
                   title=f"{CLS_LABELS[cls]}: mean coverage",
                   ylabel="log₁₀(mean coverage)")

    fig.suptitle("Per-ORF mean Ribo-seq coverage (log scale, >0 only)", fontsize=12)
    plt.tight_layout()
    fig.savefig(OUT_DIR / "coverage_distribution.png", dpi=180)
    plt.show()
    print("Saved coverage_distribution.png")

In [ ]:
if scored.empty:
    print("No scored ORFs available — skipping periodicity plot.")
else:
    tools_present = [t for t in TOOL_ORDER if t in scored['source_tool'].values]

    fig, axes = plt.subplots(1, len(CLS_ORDER), figsize=(13, 6))

    for ax, cls in zip(axes, CLS_ORDER):
        sub = scored[scored['cls'] == cls]
        positions = list(range(1, len(tools_present) + 1))

        for pos, t in zip(positions, tools_present):
            data = sub[sub['source_tool'] == t]['periodicity'].dropna().values
            if len(data) < 5:
                continue
            color = TOOL_COLORS.get(t, '#888')
            parts = ax.violinplot([data], positions=[pos], widths=0.7,
                                  showmedians=True, showextrema=False)
            for pc in parts['bodies']:
                pc.set_facecolor(color)
                pc.set_alpha(0.7)
            parts['cmedians'].set_color('black')
            parts['cmedians'].set_linewidth(2)

        ax.axhline(1/3, color='red', linestyle='--', lw=1.2, label='Random (0.33)')
        ax.set_xticks(positions)
        ax.set_xticklabels(tools_present, rotation=30, ha='right')
        ax.set_ylabel("Periodicity score (frame-0 fraction)")
        ax.set_ylim(0, 1.05)
        ax.set_title(f"{CLS_LABELS[cls]}: 3-nt periodicity")
        ax.legend(loc='upper right', fontsize=8)
        ax.yaxis.grid(True, alpha=0.4)
        ax.set_axisbelow(True)

    fig.suptitle("3-nt periodicity score (higher = stronger translation signal)", fontsize=12)
    plt.tight_layout()
    fig.savefig(OUT_DIR / "periodicity_distribution.png", dpi=180)
    plt.show()
    print("Saved periodicity_distribution.png")

In [ ]:
if scored.empty:
    print("No scored ORFs available — skipping cross-tool signal comparison.")
else:
    # ── Cross-tool signal comparison for shared ORFs ───────────────────────
    # ORFs found in ≥2 tools in the same sample
    shared_fks = (
        scored.groupby(['feature_key', 'sample_id'])['source_tool']
              .nunique()
              .pipe(lambda s: s[s >= 2])
              .index
    )
    shared_mask = [
        (row['feature_key'], row['sample_id']) in shared_fks
        for _, row in scored.iterrows()
    ]
    scored_shared = scored[shared_mask].copy()
    print(f"Shared-ORF scored rows (≥2 tools, same sample): {len(scored_shared):,}")

    if len(scored_shared) < 10:
        print("Too few shared ORFs for cross-tool scatter — skipping.")
    else:
        # For each pair of tools, pivot and scatter periodicity
        tool_pairs = [(TOOL_ORDER[i], TOOL_ORDER[j])
                      for i in range(len(TOOL_ORDER))
                      for j in range(i+1, len(TOOL_ORDER))]

        valid_pairs = [
            (a, b) for a, b in tool_pairs
            if a in scored_shared['source_tool'].values and b in scored_shared['source_tool'].values
        ]

        if not valid_pairs:
            print("No tool pairs with shared ORFs — skipping scatter.")
        else:
            ncols = min(3, len(valid_pairs))
            nrows = (len(valid_pairs) + ncols - 1) // ncols
            fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows),
                                     squeeze=False)

            for idx, (t1, t2) in enumerate(valid_pairs):
                ax = axes[idx // ncols][idx % ncols]
                sub1 = scored_shared[scored_shared['source_tool'] == t1][['feature_key', 'sample_id', 'periodicity']].rename(columns={'periodicity': 'per_t1'})
                sub2 = scored_shared[scored_shared['source_tool'] == t2][['feature_key', 'sample_id', 'periodicity']].rename(columns={'periodicity': 'per_t2'})
                merged = sub1.merge(sub2, on=['feature_key', 'sample_id']).dropna()

                if len(merged) < 3:
                    ax.set_visible(False)
                    continue

                ax.scatter(merged['per_t1'], merged['per_t2'],
                           alpha=0.4, s=10, color='#555')
                ax.plot([0, 1], [0, 1], 'r--', lw=1)
                ax.axhline(1/3, color='grey', lw=0.7, linestyle=':')
                ax.axvline(1/3, color='grey', lw=0.7, linestyle=':')
                corr = merged['per_t1'].corr(merged['per_t2'])
                ax.set_xlabel(f"{t1} periodicity")
                ax.set_ylabel(f"{t2} periodicity")
                ax.set_title(f"{t1} vs {t2}\n(r={corr:.2f}, n={len(merged):,})")
                ax.set_xlim(0, 1)
                ax.set_ylim(0, 1)

            # Hide unused subplots
            for idx in range(len(valid_pairs), nrows * ncols):
                axes[idx // ncols][idx % ncols].set_visible(False)

            fig.suptitle("Cross-tool periodicity comparison for shared feature_keys", fontsize=12)
            plt.tight_layout()
            fig.savefig(OUT_DIR / "cross_tool_periodicity_scatter.png", dpi=180)
            plt.show()
            print("Saved cross_tool_periodicity_scatter.png")

In [ ]:
if scored.empty:
    print("No scored ORFs available — skipping CDS recall vs signal plot.")
else:
    # ── CDS recall vs signal ───────────────────────────────────────────────
    # Load reference CDS feature_keys
    con = sqlite3.connect(DB)
    ref_fks = set(pd.read_sql_query("SELECT feature_key FROM reference_cds", con)['feature_key'])
    con.close()

    # Among scored CDS-class ORFs, classify as recalled vs not
    scored_cds = scored[scored['cls'] == 'cds'].copy()
    scored_cds['recalled'] = scored_cds['feature_key'].isin(ref_fks)

    recalled_cov    = scored_cds[scored_cds['recalled'] & (scored_cds['mean_cov'] > 0)]['mean_cov'].dropna()
    not_recalled_cov = scored_cds[~scored_cds['recalled'] & (scored_cds['mean_cov'] > 0)]['mean_cov'].dropna()

    print(f"Scored CDS ORFs: recalled={len(recalled_cov):,}, not-recalled={len(not_recalled_cov):,}")

    fig, ax = plt.subplots(figsize=(8, 5))
    bins = np.logspace(-2, 4, 60)

    ax.hist(recalled_cov,     bins=bins, color='#2196F3', alpha=0.65,
            label=f"Recalled (n={len(recalled_cov):,})",
            density=True)
    ax.hist(not_recalled_cov, bins=bins, color='#E53935', alpha=0.65,
            label=f"Not recalled (n={len(not_recalled_cov):,})",
            density=True)

    ax.set_xscale('log')
    ax.set_xlabel("Mean coverage (reads per nt)")
    ax.set_ylabel("Density")
    ax.set_title("CDS mean coverage: reference CDS recalled vs not-recalled\n"
                 "(exact feature_key match, any tool)")
    ax.legend()
    ax.yaxis.grid(True, alpha=0.4)
    ax.set_axisbelow(True)
    plt.tight_layout()
    fig.savefig(OUT_DIR / "cds_recall_vs_coverage.png", dpi=180)
    plt.show()
    print("Saved cds_recall_vs_coverage.png")

    # Summarise median coverage for the two groups
    print(f"\nMedian coverage — recalled: {np.median(recalled_cov):.3f} | "
          f"not-recalled: {np.median(not_recalled_cov):.3f}")
    print(f"\nAll figures saved to: {OUT_DIR.resolve()}")